In [1]:
import numpy as np
from sklearn.cluster import DBSCAN, KMeans
#import yolo
from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator
import torch 
#All ROS
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import Image, CameraInfo, PointCloud2
import sensor_msgs_py.point_cloud2 as pc2
from cv_bridge import CvBridge
from message_filters import Subscriber, ApproximateTimeSynchronizer
import ros2_numpy
from image_geometry import PinholeCameraModel


import cv2
import matplotlib.pyplot as plt
import imutils
from scipy import ndimage as nd


/home/hayashi/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
# init cuda and yolo
device = torch.device(f'cuda:{0}' if torch.cuda.is_available() else 'cpu')
model = YOLO("/home/hayashi/Desktop/yuzu_proj/best_yuzu_seg.pt").to(device)

In [ ]:
class YUZU_pointcloud(Node):
    def __init__(self):
        super().__init__('YUZU_pointcloud')
        self.camera_info_received = False
        self.bridge = CvBridge()
        self.color_frame = None
        self.color_sub = Subscriber(self, Image, "/camera/camera/color/image_rect_raw")
        self.camera_info_sub = self.create_subscription(CameraInfo,"/camera/camera/color/camera_info", self.camera_info_callback, 10)
        self.color_sub.registerCallback(self.color_callback)
        self.camera_model = PinholeCameraModel()
        self.camera_info_received = False
        
        self.depth_at_center =  0.22
        self.depth_at_edge = 0.25
        self.move_dist = 0.003

    def color_callback(self, color_msg):
        self.color_frame = self.bridge.imgmsg_to_cv2(color_msg, desired_encoding="bgr8")
        self.process()

    def camera_info_callback(self, msg):
        if not self.camera_info_received:
           self.camera_model.fromCameraInfo(msg)
           self.camera_info_received = True
           self.get_logger().info("Camera intrinsics loaded.")                                             
        
    def process(self):
        masks = None
        if self.color_frame is None:
            return

        im0 = cv2.resize(self.color_frame.copy(), (848, 480))
        #im0 = cv2.VideoCapture("yuzu.MOV")
        results = model(im0, verbose=False)
        result = results[0]
        clss = results[0].boxes.cls.cpu().tolist() if results[0].boxes is not None else []
        confs = results[0].boxes.conf.cpu().tolist() if results[0].boxes is not None else []
        boxes = results[0].boxes.xyxy.cpu().numpy() if results[0].boxes is not None else []
        annotator = Annotator(im0, line_width=2)
        confidence_text = "yuzu"
        txt_color = (255, 255, 255)
        confidence_threshold = 0.8

        if result.masks is not None:
         for i, seg in enumerate(result.masks.xy):
           if len(confs) > i and confs[i] < confidence_threshold:
                   continue
           moved_point_list = []
           seg = seg.astype(np.int32)
           seg = seg.reshape((-1, 1, 2))
           ellipse = cv2.fitEllipse(seg)
           (xc, yc), (d1, d2), angle_deg = ellipse
           a = d1/2
           b = d2/2
           axes = (int(a), int(b))
           angle = int(angle_deg)
           centroid = (int(xc), int(yc))
           point_on_curve = cv2.ellipse2Poly(centroid, axes, angle, 0, 360, 30)
           if self.camera_info_received:
               ray = self.camera_model.projectPixelTo3dRay((float(xc), float(yc)))
               real_world_point_m = [c * self.depth_at_center for c in ray] 
               real_x_m, real_y_m, real_z_m = real_world_point_m
               center_3d = np.array([real_x_m, real_y_m, real_z_m])
               for p in point_on_curve:
                   px, py = p
                   ray2 = self.camera_model.projectPixelTo3dRay((float(px), float(py)))
                   point_3d = [c * self.depth_at_edge for c in ray2]
                   vector = point_3d - center_3d
                   curr_dist = np.linalg.norm(vector)
                   if curr_dist > self.move_dist:
                       unit_vec = vector / curr_dist
                       p_moved_3d = point_3d - (unit_vec * self.move_dist)
                       px_moved, py_moved = self.camera_model.project3dToPixel(p_moved_3d)
                       moved_point_list.append([px_moved, py_moved])
               moved_points_array = np.array(moved_point_list).astype(np.float32)
               mini_ellipse = cv2.fitEllipse(moved_points_array)
               cv2.ellipse(im0, mini_ellipse,(255, 0, 255),1)
                
                
           cv2.ellipse(im0, ellipse,(0, 255, 255),1)
           cv2.polylines(im0, [seg], True, (0, 0, 255), 1)
           cv2.circle(im0, centroid, 5, (255, 0, 0), -1)


             #print(dot_y)
        cv2.imshow("detection", im0)
        key = cv2.waitKey(1)

def main(args=None):
    rclpy.init(args=args)
    node = YUZU_pointcloud()
    rclpy.spin(node)
    node.destroy_node()
    rclpy.shutdown()
    cv2.destroyAllWindows()
    node.vis.destroy_window()

if __name__ == "__main__":
    main() 

[INFO] [1769072052.921967072] [YUZU_pointcloud]: Camera intrinsics loaded.
